# Решения: Игры win/lose и инженерный выбор DP

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import csv


def find_data(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден")


def load_coin_cases():
    rows = []
    with find_data("coin_change_cases.csv").open(encoding="utf-8") as file:
        for row in csv.DictReader(file):
            rows.append((
                row["case_id"],
                int(row["amount"]),
                [int(value) for value in row["coins"].split()],
                int(row["expected_min_coins"]),
            ))
    return rows


def load_grid():
    with find_data("route_cost_grid_4x5.csv").open(encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        return [[int(value) for value in row] for row in reader]


COIN_CASES = load_coin_cases()
ROUTE_GRID = load_grid()
assert len(COIN_CASES) == 5
assert len(ROUTE_GRID) == 4 and len(ROUTE_GRID[0]) == 5


## Урок. 1. Разметить малые позиции вручную

Позиция проигрышная, если любой допустимый ход ведёт в выигрышную. Заполните значения для ходов 1, 3 и 4.

In [ ]:
moves = [1, 3, 4]
manual = [False]
for stones in range(1, 8):
    manual.append(any(stones >= move and not manual[stones - move] for move in moves))
assert all(isinstance(value, bool) for value in manual)
assert manual[0] is False
assert manual[1] is True and manual[2] is False


## Урок. 2. Таблица выигрышных позиций

Реализуйте общий алгоритм для любого положительного набора ходов.

In [ ]:
def win_table(stones, moves):
    win = [False] * (stones + 1)
    for current in range(1, stones + 1):
        win[current] = any(
            current >= move and not win[current - move]
            for move in moves
        )
    return win


table = win_table(10, [1, 3, 4])
assert len(table) == 11
assert table[:3] == [False, True, False]
assert table[10] is True


## Урок. 3. Ответ для одной позиции

`can_win` использует таблицу и возвращает только состояние, которое запросил диспетчер.

In [ ]:
def can_win(stones, moves):
    return win_table(stones, moves)[stones]


assert can_win(1, [1, 3, 4]) is True
assert can_win(2, [1, 3, 4]) is False
assert can_win(10, [1, 3, 4]) is True


## Урок. 4. Найти выигрышный ход

Верните ход в проигрышную позицию соперника. Если позиция проигрышная, верните `None`.

In [ ]:
def winning_move(stones, moves):
    table = win_table(stones, moves)
    for move in moves:
        if stones >= move and not table[stones - move]:
            return move
    return None


assert winning_move(2, [1, 3, 4]) is None
move = winning_move(10, [1, 3, 4])
assert move in [1, 3, 4]
assert can_win(10 - move, [1, 3, 4]) is False


## Урок. 5. Симуляция стратегии

Первый игрок использует `winning_move`, второй берёт первый допустимый ход. Верните журнал `(игрок, ход, остаток)`.

In [ ]:
def play_game(stones, moves):
    log = []
    player = 1
    while stones > 0:
        move = winning_move(stones, moves)
        if move is None:
            move = next(candidate for candidate in moves if candidate <= stones)
        stones -= move
        log.append((player, move, stones))
        player = 2 if player == 1 else 1
    return log


log = play_game(10, [1, 3, 4])
assert log
assert log[-1][2] == 0
assert all(move in [1, 3, 4] for _, move, _ in log)


## Урок. 6. Эксперимент: период проигрышных позиций

Для двух наборов ходов выпишите проигрышные позиции до 40. Сравните разности между соседними позициями.

In [ ]:
losing_134 = [i for i, value in enumerate(win_table(40, [1, 3, 4])) if not value]
losing_125 = [i for i, value in enumerate(win_table(40, [1, 2, 5])) if not value]
PATTERN_NOTE = (
    "На конечном диапазоне проигрышные позиции образуют повторяющийся рисунок, "
    "но таблица до 40 не является доказательством периода для всех n. Набор "
    "разрешённых ходов меняет переход и вместе с ним расположение проигрышных состояний."
)
assert losing_134[0] == 0 and losing_125[0] == 0
assert len(losing_134) >= 5 and len(losing_125) >= 5
assert len(PATTERN_NOTE) >= 120


## Урок. 7. Чек-лист применимости DP

Для задачи размена отметьте четыре проверяемых условия. Рядом с каждым булевым значением запишите конкретное обоснование.

In [ ]:
dp_fit = {
    "compact_state": (True, "Состояние полностью задаётся текущей суммой от 0 до amount."),
    "repeated_subproblems": (True, "Одна остаточная сумма возникает после выбора разных предыдущих монет."),
    "clear_transition": (True, "Ответ для суммы сравнивает dp[sum-coin] + 1 по номиналам."),
    "known_order": (True, "Суммы заполняются от 0 вверх, зависимости уже рассчитаны."),
}
assert all(flag is True for flag, _ in dp_fit.values())
assert all(len(reason) >= 40 for _, reason in dp_fit.values())


## Урок. 8. DP или более простой метод

Классифицируйте четыре задачи. Используйте только `dp`, `greedy` или `direct`; затем объясните один спорный выбор.

In [ ]:
choices = {
    "min_coins_arbitrary": "dp",
    "sum_all_costs": "direct",
    "take_largest_until_full": "greedy",
    "grid_min_path": "dp",
}
CHOICE_NOTE = (
    "Для произвольных номиналов локальный выбор крупнейшей монеты может потерять "
    "глобальный минимум, поэтому нужен DP. Простая сумма не имеет конкурирующих "
    "решений и таблица только усложнит код. Жадный метод допустим там, где условие "
    "задачи прямо требует брать крупнейшие доступные элементы."
)
assert set(choices.values()) <= {"dp", "greedy", "direct"}
assert len(set(choices.values())) == 3
assert len(CHOICE_NOTE) >= 120


## Урок. 9. Самостоятельно: артефакт инженерного выбора

Соберите функцию, которая по краткому описанию признаков задачи возвращает рекомендацию и список причин. Это не универсальный автоклассификатор, а явный чек-лист модуля.

In [ ]:
def recommend_dp(compact_state, repeated, clear_transition, ordered):
    checks = [
        ("состояние компактно", compact_state),
        ("подзадачи повторяются", repeated),
        ("переход определён", clear_transition),
        ("порядок вычисления известен", ordered),
    ]
    failed = [text for text, passed in checks if not passed]
    if failed:
        return False, failed
    return True, [text for text, _ in checks]


decision, reasons = recommend_dp(True, True, True, True)
assert decision is True
assert len(reasons) == 4
decision, reasons = recommend_dp(False, True, True, False)
assert decision is False
assert len(reasons) >= 1


## ДЗ. A1. Игра с ходами 1, 2 и 5

Верните таблицу позиций от 0 до `stones`.

In [ ]:
def game_table(stones, moves):
    table = [False] * (stones + 1)
    for current in range(1, stones + 1):
        table[current] = any(
            current >= move and not table[current - move]
            for move in moves
        )
    return table


table = game_table(12, [1, 2, 5])
assert len(table) == 13
assert table == [False, True, True, False, True, True, False, True, True, False, True, True, False]


## ДЗ. A2. Все выигрышные ходы

Верните список всех ходов, оставляющих сопернику проигрышную позицию.

In [ ]:
def all_winning_moves(stones, moves):
    table = game_table(stones, moves)
    return [
        move for move in moves
        if move <= stones and not table[stones - move]
    ]


assert all_winning_moves(2, [1, 3, 4]) == []
assert all_winning_moves(10, [1, 3, 4]) == [1, 3]


## ДЗ. A3. Инженерная нота

Сравните DP с прямым вычислением и жадным выбором на двух задачах. Укажите состояние, переход, порядок и альтернативу.

In [ ]:
ENGINEERING_NOTE = (
    "Для размена произвольными номиналами состояние — текущая сумма, переход "
    "сравнивает dp[сумма - монета] + 1, а порядок идёт от меньших сумм к большим. "
    "Жадная альтернатива проще, но номиналы 1, 3, 4 дают контрпример на сумме 6: "
    "жадный выбор использует 4 + 1 + 1, а DP находит 3 + 3. "
    "Для суммы всех стоимостей DP не нужен: нет конкурирующих решений и "
    "повторяющихся подзадач. Состояние и переход пришлось бы придумать искусственно, "
    "порядок обычного цикла уже достаточен. Альтернатива — один прямой проход по "
    "списку, который короче, прозрачнее и требует постоянной дополнительной памяти."
)
required = ["состояни", "переход", "поряд", "альтернатив"]
assert len(ENGINEERING_NOTE) >= 400
assert all(word in ENGINEERING_NOTE.lower() for word in required)
print(ENGINEERING_NOTE)


## ДЗ. B1. Период — гипотеза и проверка

Для ходов 1, 3, 4 найдите кратчайший период последних 60 значений таблицы до 200. Это вычислительная гипотеза, не доказательство.

In [ ]:
def suffix_period(values, suffix_length):
    suffix = values[-suffix_length:]
    for period in range(1, suffix_length // 2 + 1):
        if all(suffix[i] == suffix[i - period] for i in range(period, len(suffix))):
            return period
    return suffix_length


table = game_table(200, [1, 3, 4])
period = suffix_period(table, 60)
assert period == 7
